# 14. Blocked factorial inference and decision thresholds

![Paired inference](../images/14_paired_inference.svg)

**Learning goals:** preserve eight paired four-cell model blocks, calculate simple effects and the difference-in-differences interaction, use Student $t$ intervals over blocks, compare raw and clipped-logit interactions, and distinguish materiality, superiority, equivalence, and descriptive intersection gates.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

SEED = 14
rng = np.random.Generator(np.random.PCG64(SEED))
np.set_printoptions(precision=3, suppress=True)
print(f"NumPy {np.__version__}; seed={SEED}")

## 1. Eight blocks carry all four experimental cells

The score tensor has shape `(block, sequence_support, window_policy, participant)`. Index 0 and 1 on the support axis mean low and high. Index 0 and 1 on the policy axis mean frozen and resampled. Every block contains all four trained models, and the same 308 participants are evaluated in every cell. Participants sharpen each cell mean, but the primary model-level sample size is eight.

In [ ]:
B, P = 8, 308
cell_expectation = np.array([[0.42, 0.56], [0.58, 0.60]])  # [support, policy]
block_shift = rng.normal(0, 0.012, size=(B, 1, 1, 1))
block_cell_noise = rng.normal(0, 0.008, size=(B, 2, 2, 1))
participant_shift = rng.normal(0, 0.06, size=(1, 1, 1, P))
measurement_noise = rng.normal(0, 0.025, size=(B, 2, 2, P))
scores = np.clip(cell_expectation[None, :, :, None] + block_shift + block_cell_noise
                 + participant_shift + measurement_noise, 0, 1)
assert scores.shape == (8, 2, 2, 308)
cell_means = scores.mean(axis=-1)
assert cell_means.shape == (8, 2, 2)
print("score shape:", scores.shape, "cell-mean shape:", cell_means.shape)

## 2. Simple effects, interaction, and allocation contrast

For each block, $T_L$ and $T_H$ compare resampled with frozen windows at low and high sequence support. $S_F$ and $S_R$ compare high with low sequence support under each temporal policy. The primary interaction is $I=(Y_{H,R}-Y_{L,R})-(Y_{H,F}-Y_{L,F})=T_H-T_L=S_R-S_F$. A negative value means resampling helps more at low support. The direct contrast $A=Y_{L,R}-Y_{H,F}$ compares two allocations without claiming equal information or support.

In [ ]:
LF, LR = cell_means[:, 0, 0], cell_means[:, 0, 1]
HF, HR = cell_means[:, 1, 0], cell_means[:, 1, 1]
effects = {
    "T_L": LR - LF,
    "T_H": HR - HF,
    "S_F": HF - LF,
    "S_R": HR - LR,
}
interaction = (HR - LR) - (HF - LF)
allocation = LR - HF
assert np.allclose(interaction, effects["T_H"] - effects["T_L"])
assert np.allclose(interaction, effects["S_R"] - effects["S_F"])
print("mean simple effects:", {k: round(v.mean(), 3) for k, v in effects.items()})
print(f"mean I={interaction.mean():.3f}; mean A={allocation.mean():.3f}")

## 3. Student $t$ intervals use eight block contrasts

Every interval below is computed from eight within-block values and has seven degrees of freedom. The exact materiality rule has two parts: the 95% interval excludes zero and the point estimate reaches the frozen margin. The interval does not need to lie entirely beyond that margin. Equivalence instead uses a 90% interval entirely inside the declared band.

In [ ]:
def t_interval(values, confidence=0.95):
    values = np.asarray(values, dtype=np.float64)
    mean = values.mean()
    se = values.std(ddof=1) / np.sqrt(len(values))
    critical = stats.t.ppf((1 + confidence) / 2, df=len(values) - 1)
    return mean, (mean - critical * se, mean + critical * se)

def materially_positive(values, margin):
    mean, interval = t_interval(values, 0.95)
    return interval[0] > 0 and mean >= margin

def materially_negative(values, margin):
    mean, interval = t_interval(values, 0.95)
    return interval[1] < 0 and mean <= -margin

def equivalent(values, margin):
    _, interval = t_interval(values, 0.90)
    return interval[0] > -margin and interval[1] < margin

delta_t = delta_i = delta_a = 0.0625
summary = {name: (*t_interval(values),) for name, values in {**effects, "I": interaction, "A": allocation}.items()}
assert len(interaction) == 8
print("I mean and 95% interval:", summary["I"])
print("A equivalent by 90% TOST interval:", equivalent(allocation, delta_a))

## 4. Raw and clipped-logit interactions

Top-1 is bounded, so ceiling compression can change an additive interaction. The clipping constant is $1/(2\cdot308\cdot16)$. The raw percentage-scale interaction remains the sole confirmatory test. A negative raw interaction that becomes positive after the transform is scale dependent and cannot support substitution.

In [ ]:
epsilon = 1 / (2 * 308 * 16)
def clipped_logit(values):
    clipped = np.clip(values, epsilon, 1 - epsilon)
    return np.log(clipped / (1 - clipped))

z = clipped_logit(cell_means)
logit_interaction = (z[:, 1, 1] - z[:, 0, 1]) - (z[:, 1, 0] - z[:, 0, 0])
raw_interaction_summary = t_interval(interaction, 0.95)
logit_interaction_summary = t_interval(logit_interaction, 0.95)
scale_example = np.array([[0.04, 0.34], [0.78, 0.98]])
raw_reversal = (scale_example[1, 1] - scale_example[0, 1]) - (scale_example[1, 0] - scale_example[0, 0])
example_z = clipped_logit(scale_example)
logit_reversal = (example_z[1, 1] - example_z[0, 1]) - (example_z[1, 0] - example_z[0, 0])
assert raw_reversal < 0 < logit_reversal
print("raw I mean and 95% interval:", raw_interaction_summary)
print("logit I mean and 95% interval:", logit_interaction_summary)
print(f"sign-reversal example: raw={raw_reversal:.3f}, logit={logit_reversal:.3f}")

## 5. Descriptive intersection gates

Substitution-compatible evidence requires materially positive $T_L$ and $S_F$, no material harm for $T_H$ and $S_R$, a materially negative raw interaction, and the same negative sign in the clipped-logit sensitivity. Full replacement additionally requires equivalence of $A$. These are intersection labels, not extra confirmatory hypotheses. Report every component even when a gate fails.

In [ ]:
def no_material_harm(values, margin):
    _, interval = t_interval(values, 0.95)
    return interval[0] > -margin

substitution_compatible = (
    materially_positive(effects["T_L"], delta_t)
    and materially_positive(effects["S_F"], delta_t)
    and no_material_harm(effects["T_H"], delta_t)
    and no_material_harm(effects["S_R"], delta_t)
    and materially_negative(interaction, delta_i)
    and logit_interaction.mean() < 0
)
full_replacement = substitution_compatible and equivalent(allocation, delta_a)
partial_replacement = substitution_compatible and materially_negative(allocation, delta_a)
temporal_exceeds_sequence = substitution_compatible and materially_positive(allocation, delta_a)
assert sum([full_replacement, partial_replacement, temporal_exceeds_sequence]) <= 1
print({"substitution_compatible": substitution_compatible,
       "full_replacement": full_replacement,
       "partial_replacement": partial_replacement,
       "temporal_exceeds_sequence": temporal_exceeds_sequence})

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
x = np.arange(B)
ax.axhline(0, color="black", linewidth=1)
ax.plot(x, interaction, "o-", label="raw interaction")
ax.set(xlabel="paired model block", ylabel="top-1 interaction",
       title="All eight confirmatory interaction values")
ax.legend()
plt.tight_layout()
plt.show()

## Exercises and takeaways

1. Independently permute one cell's block order and recompute $I$. Which pairing is lost?
2. Change the scale-reversal example away from the boundaries. Why does reversal become harder?
3. Compare a 95% interval that excludes zero with a 90% interval inside a margin. Which supports materiality, and which supports equivalence?
4. Remove one component of the substitution gate. Why would the resulting label no longer match the frozen rule?

**Takeaway:** the primary evidence is one raw interaction per complete trained-model block. Participants improve cell measurement, but they do not change the model-level $n=8$. Materiality uses an interval and a point-estimate threshold, equivalence uses TOST, and intersection labels remain descriptive.

## Continue learning

[Previous notebook: 13](13_context_interventions.ipynb) | [Lecture](../lectures/14_paired_inference.md) | [Curriculum](../README.md) | [Next notebook: 15](15_exposure_and_replication.ipynb)